# P-Task4 (Part A): E-Commerce Inventory Analyzer

**Problem:** shop owner ki product list messy hai (duplicates, missing prices, mixed categories).
Data clean karna hai, category ke hisaab se group karna hai, har category ke top 5 mehnge items nikalne hain
aur summary dictionary banani hai. Poora kaam **comprehensions** se (Pythonic style).

**Concepts used:** list / dict / set comprehensions, conditional comprehensions, nested comprehensions,
`sorted()` with `key` and `reverse`, `zip()`, `enumerate()`.

## 1. Raw data

In [1]:
raw = [
    ("P01", "Laptop Stand", "Electronics", 2500), ("P02", "Mouse", "electronics", 1200),
    ("P03", "Keyboard", "Electronics ", 3500), ("P04", "Webcam", "Electronics", 4800),
    ("P05", "Headphones", "ELECTRONICS", 6500), ("P06", "Power Bank", "Electronics", 3900),
    ("P07", "Monitor", "Electronics", 28000), ("P08", "USB Hub", "Electronics", None),
    ("P09", "T-Shirt", "Clothing", 900), ("P10", "Jeans", "clothing", 2800),
    ("P11", "Jacket", "Clothing", 5500), ("P12", "Socks", "Clothing", 250),
    ("P13", "Cap", "Clothing", 450), ("P14", "Sweater", "Clothing", 3200),
    ("P15", "Scarf", "Clothing", 700), ("P16", "Shoes", "Footwear", 4200),
    ("P17", "Sandals", "footwear", 1500), ("P18", "Sneakers", "Footwear", 6800),
    ("P19", "Mouse", "electronics", 1200),                         # duplicate
    ("P20", "Novel", "Books", 800), ("P21", "Python Book", "Books", 1800),
    ("P22", "Dictionary", "Books", ""), ("P23", "Notebook", "Books", 150),
    ("P24", "T-Shirt", "Clothing", 900),                           # duplicate
]
len(raw)

24

## 2. Cleaning (comprehensions se)
* price missing (`None` ya khali string) wali rows hata do
* category ko `strip().title()` karo
* duplicate (same name + category) hata do, set comprehension se pehle unique keys nikaali

In [2]:
# step 1: sirf wo rows jin ki price hai
with_price = [r for r in raw if r[3] not in (None, "")]

# step 2: category / name ko theek karke dict ki list
tidy = [{"id": pid, "name": name.strip(), "category": cat.strip().title(), "price": float(price)}
        for pid, name, cat, price in with_price]

# step 3: duplicates. (name, category) jo pehli baar aaya wohi rakho
unique_keys = {(p["name"], p["category"]) for p in tidy}
first_seen = {}
for p in tidy:
    first_seen.setdefault((p["name"], p["category"]), p)
clean = list(first_seen.values())

print("raw:", len(raw), "| price wali:", len(with_price), "| unique:", len(unique_keys), "| clean:", len(clean))

raw: 24 | price wali: 22 | unique: 20 | clean: 20


## 3. Category ke hisaab se group
Dict comprehension, andar nested list comprehension.

In [3]:
categories = sorted({p["category"] for p in clean})

grouped = {cat: [p for p in clean if p["category"] == cat] for cat in categories}
{cat: len(items) for cat, items in grouped.items()}

{'Books': 3, 'Clothing': 7, 'Electronics': 7, 'Footwear': 3}

## 4. Har category ke top 5 mehnge items

In [4]:
top5 = {
    cat: sorted(items, key=lambda p: p["price"], reverse=True)[:5]
    for cat, items in grouped.items()
}

for cat, items in top5.items():
    print(cat)
    for rank, p in enumerate(items, start=1):
        print(f"   {rank}. {p['name']:<14} Rs {p['price']:>9,.0f}")

Books
   1. Python Book    Rs     1,800
   2. Novel          Rs       800
   3. Notebook       Rs       150
Clothing
   1. Jacket         Rs     5,500
   2. Sweater        Rs     3,200
   3. Jeans          Rs     2,800
   4. T-Shirt        Rs       900
   5. Scarf          Rs       700
Electronics
   1. Monitor        Rs    28,000
   2. Headphones     Rs     6,500
   3. Webcam         Rs     4,800
   4. Power Bank     Rs     3,900
   5. Keyboard       Rs     3,500
Footwear
   1. Sneakers       Rs     6,800
   2. Shoes          Rs     4,200
   3. Sandals        Rs     1,500


## 5. Summary dictionary

In [5]:
summary = {
    cat: {
        "count": len(items),
        "total_value": sum(p["price"] for p in items),
        "average": round(sum(p["price"] for p in items) / len(items), 2),
        "cheapest": min(items, key=lambda p: p["price"])["name"],
        "most_expensive": max(items, key=lambda p: p["price"])["name"],
    }
    for cat, items in grouped.items()
}

for cat, info in summary.items():
    print(cat, "->", info)

Books -> {'count': 3, 'total_value': 2750.0, 'average': 916.67, 'cheapest': 'Notebook', 'most_expensive': 'Python Book'}
Clothing -> {'count': 7, 'total_value': 13800.0, 'average': 1971.43, 'cheapest': 'Socks', 'most_expensive': 'Jacket'}
Electronics -> {'count': 7, 'total_value': 50400.0, 'average': 7200.0, 'cheapest': 'Mouse', 'most_expensive': 'Monitor'}
Footwear -> {'count': 3, 'total_value': 12500.0, 'average': 4166.67, 'cheapest': 'Sandals', 'most_expensive': 'Sneakers'}


## 6. Thode extra checks (`zip`, `map`, `filter`)

In [6]:
names = [p["name"] for p in clean]
prices = [p["price"] for p in clean]
name_price = dict(zip(names, prices))

expensive = list(filter(lambda kv: kv[1] > 5000, name_price.items()))
print("5000 se mehnge items:", expensive)

with_tax = list(map(lambda x: round(x * 1.17, 2), prices[:5]))
print("pehle 5 prices 17% tax ke saath:", with_tax)

5000 se mehnge items: [('Headphones', 6500.0), ('Monitor', 28000.0), ('Jacket', 5500.0), ('Sneakers', 6800.0)]
pehle 5 prices 17% tax ke saath: [2925.0, 1404.0, 4095.0, 5616.0, 7605.0]


## Conclusion
Poora cleaning, grouping aur summary ka kaam `for` loop ke bajaye comprehensions se kiya. Code chhota bhi hai
aur parhne mein bhi saaf lagta hai. Duplicate aur missing price wali rows nikal gayin aur har category ke top 5
mehnge items mil gaye.